# Phase 4 Final — Phase 2 top-80 + Vietnamese Legal BGE + protected rank blender

Attach the competition data, Phase 2 Harrier bundle, Phase 3 reranker delta bundle, and the saved output of the completed Phase 2 private notebook. Select RTX Pro 6000 and set Internet to Off.

The notebook reuses every expensive Phase 2 retrieval/Jina/Vietnamese-reranker artifact. It runs only the small Vietnamese legal BGE reranker over the existing top-80 candidates, then trains a cross-fitted CPU rank blender with protected promotion from the proven Phase 2 top five. Previous submissions are never modified.


In [1]:
import os
import json

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from pathlib import Path

EXPERIMENT_ID = 'phase4-phase2-legal-bge-protected-blender'

TEST_FILENAME = 'private-official.json'
if TEST_FILENAME not in {'private-official.json', 'public-official.json'}:
    raise ValueError(TEST_FILENAME)

TEST_LABEL = 'private' if TEST_FILENAME == 'private-official.json' else 'public'

INPUT_ROOT = Path('/kaggle/input')
WORK_DIR = Path(f'/kaggle/working/legalir-phase4-{TEST_LABEL}-final')
RUNTIME_DIR = Path('/kaggle/working/legalir-phase4-runtime')


def one(items, label):
    items = list(items)
    if len(items) != 1:
        raise RuntimeError(
            f'Expected exactly one {label}, found {len(items)}:\n'
            + '\n'.join(map(str, items))
        )
    return items[0]


def manifest_with_id(filename, experiment_id):
    matches = []

    for path in INPUT_ROOT.rglob(filename):
        try:
            manifest = json.loads(path.read_text(encoding='utf-8'))
        except Exception:
            continue

        if manifest.get('experiment_id') == experiment_id:
            matches.append(path)

    return one(matches, experiment_id)


# ------------------------------------------------------------
# Phase 2 bundle
# ------------------------------------------------------------

PHASE2_MANIFEST_PATH = manifest_with_id(
    'bundle_manifest.json',
    'phase2-vietlegal-harrier-0.6b',
)

PHASE2_BUNDLE = PHASE2_MANIFEST_PATH.parents[1]


# ------------------------------------------------------------
# Phase 3 reranker delta bundle
# ------------------------------------------------------------

PHASE3_MANIFEST_PATH = manifest_with_id(
    'bundle_manifest.json',
    'phase3-rerankers-harrier-retrieval',
)

PHASE3_DELTA = PHASE3_MANIFEST_PATH.parents[1]


# ------------------------------------------------------------
# Official competition dataset
# ------------------------------------------------------------

# Avoid accidentally detecting train.json inside one of the bundles.
excluded_top_level = {
    path.relative_to(INPUT_ROOT).parts[0]
    for path in (
        PHASE2_MANIFEST_PATH,
        PHASE3_MANIFEST_PATH,
    )
}

data_matches = []

for train_path in INPUT_ROOT.rglob('train.json'):
    dataset_root = train_path.parent

    if not (dataset_root / TEST_FILENAME).is_file():
        continue

    contexts = list(dataset_root.rglob('context_*.json'))
    if not contexts:
        continue

    contexts_dir = contexts[0].parent
    data_matches.append((dataset_root, contexts_dir))


preferred_matches = [
    match
    for match in data_matches
    if match[0].relative_to(INPUT_ROOT).parts[0]
    not in excluded_top_level
]

DATASET_DIR, CONTEXTS_DIR = one(
    preferred_matches or data_matches,
    'official competition dataset',
)


# Phase 2 artifacts will be detected automatically later.
PHASE2_ARTIFACTS_OVERRIDE = None


print('Input root       :', INPUT_ROOT)
print('Dataset          :', DATASET_DIR)
print('Contexts         :', CONTEXTS_DIR)
print('Phase 2 bundle   :', PHASE2_BUNDLE)
print('Phase 3 delta    :', PHASE3_DELTA)
print('Work directory   :', WORK_DIR)

Input root       : /kaggle/input
Dataset          : /kaggle/input/datasets/tonioz/uit-dsc-task1
Contexts         : /kaggle/input/datasets/tonioz/uit-dsc-task1/selected-contexts/selected-contexts
Phase 2 bundle   : /kaggle/input/datasets/boinhbo/legalir-phase2-harrier-bundle/legalir-phase2-harrier-bundle
Phase 3 delta    : /kaggle/input/datasets/boinhbo/legalir-phase3-reranker-delta/legalir-phase3-reranker-delta
Work directory   : /kaggle/working/legalir-phase4-private-final


In [2]:
import json
import shutil
import subprocess
import sys
import time

def run(*command, cwd=None, env=None):
    print('+', ' '.join(map(str, command)))
    started = time.perf_counter()
    subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)
    print(f'Completed in {(time.perf_counter() - started) / 60:.1f} minutes')

phase2_manifest = json.loads((PHASE2_BUNDLE / 'manifests' / 'bundle_manifest.json').read_text(encoding='utf-8'))
delta_manifest = json.loads((PHASE3_DELTA / 'manifests' / 'bundle_manifest.json').read_text(encoding='utf-8'))
if phase2_manifest.get('experiment_id') != 'phase2-vietlegal-harrier-0.6b':
    raise RuntimeError('Wrong Phase 2 bundle')
if delta_manifest.get('experiment_id') != 'phase3-rerankers-harrier-retrieval':
    raise RuntimeError('Wrong Phase 3 delta bundle')
if 'legal_reranker' not in {row['name'] for row in delta_manifest['models']}:
    raise RuntimeError('Phase 3 delta lacks legal_reranker')
for manifest, bundle in ((phase2_manifest, PHASE2_BUNDLE), (delta_manifest, PHASE3_DELTA)):
    for record in manifest['files']:
        path = bundle / record['path']
        if not path.is_file() or path.stat().st_size != record['bytes']:
            raise RuntimeError(f'Missing or truncated bundle file: {path}')

WORK_DIR.mkdir(parents=True, exist_ok=True)
if RUNTIME_DIR.exists():
    shutil.rmtree(RUNTIME_DIR)
RUNTIME_DIR.mkdir(parents=True)
run(sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--ignore-installed', '--target', RUNTIME_DIR, '--find-links', PHASE3_DELTA / 'wheels', '-r', PHASE3_DELTA / 'requirements-offline.txt')
project_wheels = sorted((PHASE3_DELTA / 'wheels').glob('uit_legalir-*.whl'))
if len(project_wheels) != 1:
    raise RuntimeError(f'Expected one project wheel, found {project_wheels}')
run(sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--ignore-installed', '--target', RUNTIME_DIR, project_wheels[0])
runtime_env = os.environ.copy()
runtime_env['PYTHONPATH'] = str(RUNTIME_DIR)
runtime_env['PYTHONNOUSERSITE'] = '1'
probe = "import sklearn, torch, transformers; assert torch.cuda.is_available(); print('sklearn', sklearn.__version__); print('GPU', torch.cuda.get_device_name(0)); print('transformers', transformers.__version__)"
run(sys.executable, '-c', probe, env=runtime_env)
print('Runtime project commit:', delta_manifest['project_commit'])


+ /usr/bin/python3 -m pip install --no-index --no-deps --ignore-installed --target /kaggle/working/legalir-phase4-runtime --find-links /kaggle/input/datasets/boinhbo/legalir-phase3-reranker-delta/legalir-phase3-reranker-delta/wheels -r /kaggle/input/datasets/boinhbo/legalir-phase3-reranker-delta/legalir-phase3-reranker-delta/requirements-offline.txt
Looking in links: /kaggle/input/datasets/boinhbo/legalir-phase3-reranker-delta/legalir-phase3-reranker-delta/wheels
Processing /kaggle/input/datasets/boinhbo/legalir-phase3-reranker-delta/legalir-phase3-reranker-delta/wheels/faiss_cpu-1.15.1-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (from -r /kaggle/input/datasets/boinhbo/legalir-phase3-reranker-delta/legalir-phase3-reranker-delta/requirements-offline.txt (line 4))
Processing /kaggle/input/datasets/boinhbo/legalir-phase3-reranker-delta/legalir-phase3-reranker-delta/wheels/sentence_transformers-5.7.0-py3-none-any.whl (from -r /kaggle/input/datasets/boinhbo/legalir-phase3-rer

In [3]:
import hashlib

# ------------------------------------------------------------
# Current official data / input fingerprint
# ------------------------------------------------------------

contexts_source = CONTEXTS_DIR

if not contexts_source.is_dir():
    raise FileNotFoundError(f'Missing corpus: {contexts_source}')

for filename in ('train.json', TEST_FILENAME):
    path = DATASET_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(path)


def ensure_link(destination, source, is_directory=False):
    if destination.is_symlink():
        if destination.resolve() == source.resolve():
            return
        destination.unlink()
    elif destination.exists():
        raise RuntimeError(f'Refusing to overwrite: {destination}')

    destination.symlink_to(
        source,
        target_is_directory=is_directory,
    )


# Keep the work directory compatible with the original CLI layout too.
ensure_link(
    WORK_DIR / 'selected-contexts',
    contexts_source,
    True,
)

for filename in ('train.json', TEST_FILENAME):
    ensure_link(
        WORK_DIR / filename,
        DATASET_DIR / filename,
    )


test_bytes = (DATASET_DIR / TEST_FILENAME).read_bytes()
test_sha256 = hashlib.sha256(test_bytes).hexdigest()
test_question_count = len(json.loads(test_bytes))

print(
    'Corpus documents:',
    sum(1 for _ in contexts_source.glob('context_*.json')),
)
print(
    'Test questions:',
    test_question_count,
    'sha256:',
    test_sha256,
)


# ------------------------------------------------------------
# Locate the EXACT saved Phase 2 private run by its input state.
#
# Important:
# The saved Kaggle dataset intentionally contains the expensive
# retrieval/reranker caches, but NOT corpus/chunk/dense-index
# symlink targets. Phase 4 must not require those here.
# ------------------------------------------------------------

state_matches = []

for state_path in INPUT_ROOT.rglob('inference_input_state.json'):
    try:
        state = json.loads(state_path.read_text(encoding='utf-8'))
    except Exception:
        continue

    if (
        state.get('filename') == TEST_FILENAME
        and state.get('sha256') == test_sha256
        and state.get('questions') == test_question_count
        and state.get('project_commit') == delta_manifest.get('project_commit')
    ):
        state_matches.append((state_path, state))

if len(state_matches) != 1:
    raise RuntimeError(
        'Expected exactly one saved Phase 2 run matching '
        f'{TEST_FILENAME} / {test_sha256} / '
        f'project_commit={delta_manifest.get("project_commit")}; '
        f'found {len(state_matches)}:\n'
        + '\n'.join(str(path) for path, _ in state_matches)
    )

state_path, state = state_matches[0]
PHASE2_ARTIFACTS = state_path.parent / 'artifacts_phase2_harrier'

if not PHASE2_ARTIFACTS.is_dir():
    raise FileNotFoundError(
        f'Matching Phase 2 state was found, but artifact directory is missing: '
        f'{PHASE2_ARTIFACTS}'
    )


# ------------------------------------------------------------
# Validate ONLY the files that Phase 4 actually reuses.
# ------------------------------------------------------------

required_cache_files = (
    'prepare_manifest.json',
    'first_stage_weights.json',
    'final_weights.json',
    'retrieval_train.json',
    'retrieval_public.json',
)

missing = [
    name
    for name in required_cache_files
    if not (PHASE2_ARTIFACTS / name).is_file()
    or (PHASE2_ARTIFACTS / name).stat().st_size == 0
]


def has_rerank_cache(split, engine, fold=None):
    suffix = f'_{fold}' if fold is not None else ''

    separate = (
        PHASE2_ARTIFACTS
        / f'rerank_{split}{suffix}_{engine}.json'
    )
    if separate.is_file() and separate.stat().st_size > 0:
        try:
            payload = json.loads(
                separate.read_text(encoding='utf-8')
            )
            if engine in payload:
                return True
        except Exception:
            pass

    combined = (
        PHASE2_ARTIFACTS
        / f'rerank_{split}{suffix}.json'
    )
    if combined.is_file() and combined.stat().st_size > 0:
        try:
            payload = json.loads(
                combined.read_text(encoding='utf-8')
            )
            if engine in payload:
                return True
        except Exception:
            pass

    return False


for engine in ('jina', 'vietnamese_reranker'):
    if not has_rerank_cache('train', engine, fold=0):
        missing.append(f'train fold-0 rerank cache for {engine}')

    if not has_rerank_cache('public', engine):
        missing.append(f'public rerank cache for {engine}')


if missing:
    raise RuntimeError(
        'The matching Phase 2 run exists, but required reusable '
        'Phase 2 caches are missing:\n- '
        + '\n- '.join(missing)
    )


phase2_prepare_manifest = json.loads(
    (PHASE2_ARTIFACTS / 'prepare_manifest.json')
    .read_text(encoding='utf-8')
)

print('Using Phase 2 artifacts:', PHASE2_ARTIFACTS)
print('Input state file        :', state_path)
print('Input fingerprint       :', state)
print(
    'Phase 2 prepared data   :',
    {
        key: phase2_prepare_manifest.get(key)
        for key in (
            'documents',
            'short_chunks',
            'long_chunks',
            'train_questions',
            'public_questions',
        )
    },
)


Corpus documents: 8532
Test questions: 2080 sha256: 9da4e0cb84204fed924251c35744c93879556e67a440332015ea3b62f3c355bc
Using Phase 2 artifacts: /kaggle/input/datasets/boinhbo/artifacts-phase2/legalir-phase2-private-harrier-run/artifacts_phase2_harrier
Input state file        : /kaggle/input/datasets/boinhbo/artifacts-phase2/legalir-phase2-private-harrier-run/inference_input_state.json
Input fingerprint       : {'filename': 'private-official.json', 'sha256': '9da4e0cb84204fed924251c35744c93879556e67a440332015ea3b62f3c355bc', 'questions': 2080, 'config_sha256': '01555e7035c175e0db4c2a2420c1490e29cd0e4675215fccbd397fc4531cbebd', 'project_commit': '944a51093314e299aa1f6855f0fdb8b9d2c293db'}
Phase 2 prepared data   : {'documents': 8532, 'short_chunks': 420311, 'long_chunks': 255832, 'train_questions': 7000, 'public_questions': 2080}


In [4]:
import yaml

# ------------------------------------------------------------
# Build Phase 4 config from the same Phase 2 config.
# Only the new legal reranker and Phase 4 reranking settings change.
# ------------------------------------------------------------

config = yaml.safe_load(
    (
        PHASE2_BUNDLE
        / 'configs'
        / 'kaggle_rtx_pro_6000.yaml'
    ).read_text(encoding='utf-8')
)

# Use absolute official-data paths so Phase 4 does not depend on
# hidden/saved symlink targets from the old Phase 2 working directory.
config['paths']['contexts_dir'] = str(CONTEXTS_DIR)
config['paths']['train_file'] = str(DATASET_DIR / 'train.json')
config['paths']['public_file'] = str(DATASET_DIR / TEST_FILENAME)
config['paths']['artifacts_dir'] = str(
    WORK_DIR / 'artifacts_phase4'
)

for name in (
    'vietlegal_harrier',
    'vietnamese_embedding',
    'nemotron',
    'jina',
    'vietnamese_reranker',
):
    config['models'][name]['local_path'] = str(
        PHASE2_BUNDLE / 'models' / name
    )
    config['models'][name]['local_files_only'] = True

phase3_config = yaml.safe_load(
    (
        PHASE3_DELTA
        / 'configs'
        / 'kaggle_rtx_pro_6000.yaml'
    ).read_text(encoding='utf-8')
)

config['models']['legal_reranker'] = (
    phase3_config['models']['legal_reranker']
)
config['models']['legal_reranker']['local_path'] = str(
    PHASE3_DELTA / 'models' / 'legal_reranker'
)
config['models']['legal_reranker']['local_files_only'] = True

config['reranking']['rerank_top_k'] = 80
config['reranking']['pairwise_max_length'] = 512
config['reranking']['pairwise_evidence_tokens'] = 440
config['reranking']['pairwise_batch_size'] = 64
config['reranking']['query_batch_size'] = 16
config['reranking']['checkpoint_every_questions'] = 32
config['validation']['reranker_tuning_folds'] = [0]

artifacts = Path(config['paths']['artifacts_dir'])
artifacts.mkdir(parents=True, exist_ok=True)

config_path = (
    WORK_DIR / 'kaggle_rtx_pro_6000_phase4.yaml'
)
config_path.write_text(
    yaml.safe_dump(
        config,
        allow_unicode=True,
        sort_keys=False,
    ),
    encoding='utf-8',
)


# ------------------------------------------------------------
# Rebuild ONLY the lightweight prepared corpus/questions.
#
# The Phase 2 saved-output dataset does not contain corpus.jsonl,
# chunks_*.jsonl, dense indexes, etc. Those were intermediate
# symlink/cache targets. We recreate the prepared text from the
# official source data, but DO NOT rebuild retrieval indexes.
# ------------------------------------------------------------

run(
    sys.executable,
    '-m',
    'legalir',
    'prepare',
    '--config',
    config_path,
    '--resume',
    cwd=WORK_DIR,
    env=runtime_env,
)


# ------------------------------------------------------------
# Prove that the newly prepared data is identical to the data
# used by the saved Phase 2 caches before reusing any cache.
# ------------------------------------------------------------

current_prepare_manifest = json.loads(
    (artifacts / 'prepare_manifest.json')
    .read_text(encoding='utf-8')
)

compatibility_keys = (
    'schema_version',
    'chunking_fingerprint',
    'documents',
    'short_chunks',
    'long_chunks',
    'train_questions',
    'public_questions',
    'train_questions_fingerprint',
    'public_questions_fingerprint',
)

manifest_mismatches = {
    key: {
        'phase2': phase2_prepare_manifest.get(key),
        'phase4_prepare': current_prepare_manifest.get(key),
    }
    for key in compatibility_keys
    if phase2_prepare_manifest.get(key)
    != current_prepare_manifest.get(key)
}

if manifest_mismatches:
    raise RuntimeError(
        'Current official data/preparation does not match the '
        'saved Phase 2 caches. Refusing to reuse stale caches:\n'
        + json.dumps(
            manifest_mismatches,
            ensure_ascii=False,
            indent=2,
        )
    )

print('Prepared-data fingerprint matches saved Phase 2 caches.')


# ------------------------------------------------------------
# Reuse ONLY Phase 2 caches that are safe to carry into Phase 4.
#
# CRITICAL:
# Do NOT seed the combined rerank_train_0.json / rerank_public.json.
# The legalir CLI treats those combined files as completed reranking
# checkpoints under --resume. If we seed them from Phase 2, the new
# legal_reranker is skipped entirely.
#
# We only seed the per-engine Phase 2 reranker caches (Jina + VN),
# so Phase 4 can compute its new legal_reranker independently.
# ------------------------------------------------------------

cache_patterns = (
    'first_stage_weights.json',
    'retrieval_train.json',
    'retrieval_public.json',
    'rerank_train_0_jina.json',
    'rerank_train_0_vietnamese_reranker.json',
    'rerank_public_jina.json',
    'rerank_public_vietnamese_reranker.json',
)

# Remove stale rerank outputs from any previous failed attempt.
stale_phase4_rerank = (
    'rerank_train_0.json',
    'rerank_public.json',
    'rerank_train_0_legal_reranker.json',
    'rerank_public_legal_reranker.json',
    'rerank_train_0_legal_reranker.progress.json',
    'rerank_public_legal_reranker.progress.json',
)

for name in stale_phase4_rerank:
    path = artifacts / name
    if path.is_symlink() or path.is_file():
        path.unlink()

reused = []

for name in cache_patterns:
    source = PHASE2_ARTIFACTS / name

    if not source.is_file():
        raise FileNotFoundError(
            f'Required Phase 2 reusable cache is missing: {source}'
        )

    destination = artifacts / name

    if destination.is_symlink():
        if destination.resolve() == source.resolve():
            reused.append(name)
            continue
        destination.unlink()

    elif destination.exists():
        destination.unlink()

    destination.symlink_to(source)
    reused.append(name)


# Keep the Phase 2 final weights under a distinct name because
# Phase 4's blender treats them as the protected baseline.
shutil.copy2(
    PHASE2_ARTIFACTS / 'final_weights.json',
    artifacts / 'phase2_final_weights.json',
)


# ------------------------------------------------------------
# Strict Phase 2 cache identity audit
# ------------------------------------------------------------

phase2_model_manifest_path = (
    PHASE2_ARTIFACTS / 'model_manifest.json'
)

if not phase2_model_manifest_path.is_file():
    raise FileNotFoundError(
        phase2_model_manifest_path
    )

phase2_model_manifest = json.loads(
    phase2_model_manifest_path.read_text(
        encoding='utf-8'
    )
)

phase2_models = {
    row['name']: row
    for row in phase2_model_manifest.get(
        'models',
        [],
    )
}

expected_phase2_models = (
    'vietlegal_harrier',
    'vietnamese_embedding',
    'nemotron',
    'jina',
    'vietnamese_reranker',
)

model_mismatches = {}

for name in expected_phase2_models:
    saved = phase2_models.get(name)
    current = config['models'].get(name)

    if saved is None or current is None:
        model_mismatches[name] = {
            'saved': saved,
            'current': current,
        }
        continue

    if (
        saved.get('id') != current.get('id')
        or saved.get('revision')
        != current.get('revision')
    ):
        model_mismatches[name] = {
            'saved_id': saved.get('id'),
            'saved_revision': saved.get('revision'),
            'current_id': current.get('id'),
            'current_revision': current.get(
                'revision'
            ),
        }

if model_mismatches:
    raise RuntimeError(
        'Phase 2 model identity mismatch; '
        'refusing to reuse caches:\n'
        + json.dumps(
            model_mismatches,
            ensure_ascii=False,
            indent=2,
        )
    )


# Validate reused per-engine rerank caches.
for split, fold in (
    ('train', 0),
    ('public', None),
):
    suffix = (
        f'_{fold}'
        if fold is not None
        else ''
    )

    for engine in (
        'jina',
        'vietnamese_reranker',
    ):
        path = (
            artifacts
            / f'rerank_{split}{suffix}_{engine}.json'
        )

        payload = json.loads(
            path.read_text(
                encoding='utf-8'
            )
        )

        if (
            engine not in payload
            or not payload[engine]
        ):
            raise RuntimeError(
                f'Invalid Phase 2 rerank cache: {path}'
            )

print(
    'Prepared-data fingerprint matches '
    'saved Phase 2 caches.'
)

print(
    'Phase 2 model identities match '
    'current Phase 2 configuration.'
)

print('Reused Phase 2 caches:')
for name in reused:
    print(' -', name)

print('NOT reused on purpose:')
print(' - rerank_train_0.json')
print(' - rerank_public.json')

print(
    'These must be rebuilt/extended '
    'for legal_reranker in Phase 4.'
)

print('Phase 4 artifacts:', artifacts)
print('Phase 4 config   :', config_path)

+ /usr/bin/python3 -m legalir prepare --config /kaggle/working/legalir-phase4-private-final/kaggle_rtx_pro_6000_phase4.yaml --resume


Preparing legal corpus: 100%|██████████| 8532/8532 [02:44<00:00, 51.87it/s]


{
  "schema_version": 2,
  "chunking_fingerprint": "58304a511c48e2ea202420ea24760f3591d5221d3140ffe7c258f57709e71b6c",
  "documents": 8532,
  "short_chunks": 420311,
  "long_chunks": 255832,
  "train_questions": 7000,
  "public_questions": 2080,
  "train_questions_fingerprint": "4a2f2a17b853138438c2b61c853f96637781909ce306bcd326a0213d53b4352e",
  "public_questions_fingerprint": "915600d911c468197dbf376a346767bfc527c781f3f5f7f56525898beba171e1"
}
Completed in 3.1 minutes
Prepared-data fingerprint matches saved Phase 2 caches.
Prepared-data fingerprint matches saved Phase 2 caches.
Phase 2 model identities match current Phase 2 configuration.
Reused Phase 2 caches:
 - first_stage_weights.json
 - retrieval_train.json
 - retrieval_public.json
 - rerank_train_0_jina.json
 - rerank_train_0_vietnamese_reranker.json
 - rerank_public_jina.json
 - rerank_public_vietnamese_reranker.json
NOT reused on purpose:
 - rerank_train_0.json
 - rerank_public.json
These must be rebuilt/extended for legal_re

In [5]:
# Only the newly added model needs a preflight.
# Phase 2 models already produced the attached caches.

preflight = (
    WORK_DIR / 'phase4_legal_preflight.py'
)

preflight.write_text(
    "import sys\n"
    "from pathlib import Path\n"
    "import yaml\n"
    "from legalir.rerank import PairwiseReranker\n"
    "config = yaml.safe_load("
    "Path(sys.argv[1]).read_text(encoding='utf-8'))\n"
    "engine = PairwiseReranker("
    "config, 'legal_reranker')\n"
    "order = engine.rank("
    "'điều kiện cấp giấy phép', "
    "['văn bản có quy định cấp giấy phép', "
    "'văn bản không liên quan'])\n"
    "assert sorted(order) == [0, 1]\n"
    "engine.close()\n"
    "print('Phase 4 legal reranker preflight passed')\n",
    encoding='utf-8',
)

run(
    sys.executable,
    preflight,
    config_path,
    cwd=WORK_DIR,
    env=runtime_env,
)

base = [
    sys.executable,
    '-m',
    'legalir',
]


def legalir(*args):
    run(
        *base,
        *args,
        cwd=WORK_DIR,
        env=runtime_env,
    )


# Audit deployed checkpoints, but do not
# recompute Phase 2 retrieval/reranking.
legalir(
    'audit',
    '--config',
    config_path,
)


# IMPORTANT:
# Combined Phase 2 rerank caches are NOT present here,
# so the new legal reranker must actually run.

legalir(
    'rerank',
    '--config',
    config_path,
    '--split',
    'train',
    '--fold',
    '0',
    '--engine',
    'legal_reranker',
    '--resume',
)

legalir(
    'rerank',
    '--config',
    config_path,
    '--split',
    'public',
    '--engine',
    'legal_reranker',
    '--resume',
)


# ------------------------------------------------------------
# Hard gate:
# prove Phase 4's new legal reranker caches
# were really created before blender runs.
# ------------------------------------------------------------

expected_legal_caches = (
    artifacts
    / 'rerank_train_0_legal_reranker.json',

    artifacts
    / 'rerank_public_legal_reranker.json',
)

for path in expected_legal_caches:

    if (
        not path.is_file()
        or path.stat().st_size == 0
    ):
        raise RuntimeError(
            'legal_reranker command finished '
            'but its per-engine cache was '
            f'not created: {path}'
        )

    payload = json.loads(
        path.read_text(
            encoding='utf-8'
        )
    )

    if (
        'legal_reranker' not in payload
        or not payload['legal_reranker']
    ):
        raise RuntimeError(
            f'Invalid legal_reranker cache: {path}'
        )

print(
    'Phase 4 legal reranker caches verified:'
)

for path in expected_legal_caches:
    print(' -', path)

+ /usr/bin/python3 /kaggle/working/legalir-phase4-private-final/phase4_legal_preflight.py /kaggle/working/legalir-phase4-private-final/kaggle_rtx_pro_6000_phase4.yaml


Loading weights: 100%|██████████| 201/201 [00:01<00:00, 117.60it/s]


Phase 4 legal reranker preflight passed
Completed in 0.1 minutes
+ /usr/bin/python3 -m legalir audit --config /kaggle/working/legalir-phase4-private-final/kaggle_rtx_pro_6000_phase4.yaml


Loading weights: 100%|██████████| 391/391 [00:03<00:00, 117.81it/s]
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8825.03it/s]


{
  "models": [
    {
      "name": "vietlegal_harrier",
      "id": "mainguyen9/vietlegal-harrier-0.6b",
      "parameters": 596049920,
      "revision": "91a0e1ebe4b63b4475bbae40658b8ca9231bea74",
      "license": "Apache-2.0"
    },
    {
      "name": "vietnamese_embedding",
      "id": "AITeamVN/Vietnamese_Embedding_v2",
      "parameters": 567754752,
      "revision": "18b44161e041bf1d3a333ab5144b5b7b93f914d2",
      "license": "Apache-2.0"
    },
    {
      "name": "nemotron",
      "id": "nvidia/Nemotron-3-Embed-1B-BF16",
      "parameters": 1140918272,
      "revision": "c0c9fea93ea424587517f2c59e20db9f1d6bf615",
      "license": "OpenMDW-1.1"
    },
    {
      "name": "jina",
      "id": "jinaai/jina-reranker-v3.5",
      "parameters": 596836352,
      "revision": "e8a93f33f0b22108f8c2364f8484ce3422552fbc",
      "license": "CC-BY-NC-4.0"
    },
    {
      "name": "vietnamese_reranker",
      "id": "AITeamVN/Vietnamese_Reranker",
      "parameters": 567755777,
      "revis

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8726.47it/s]


{
  "legal_reranker": {
    "79386": [
      "208565",
      "154667",
      "110305",
      "48429",
      "299574",
      "301570",
      "32994",
      "122148",
      "17545",
      "93888",
      "34537",
      "147066",
      "83446",
      "273934",
      "109796",
      "14585",
      "263368",
      "42692",
      "233902",
      "245154",
      "163407",
      "282052",
      "24778",
      "69191",
      "32778",
      "48031",
      "47138",
      "200446",
      "11898",
      "163290",
      "21111",
      "202672",
      "47682",
      "258723",
      "274327",
      "299073",
      "94717",
      "35726",
      "171342",
      "90941",
      "15102",
      "191772",
      "258706",
      "129559",
      "54146",
      "60783",
      "261198",
      "194094",
      "78507",
      "98575",
      "99233",
      "66112",
      "6525",
      "211965",
      "96584",
      "30220",
      "11173",
      "5051",
      "289767",
      "71886",
      "136960",
      "220359",
   

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8427.35it/s]


{
  "legal_reranker": {
    "44474": [
      "26173",
      "115380",
      "55018",
      "292631",
      "222640",
      "155884",
      "253656",
      "200618",
      "76646",
      "97924",
      "125251",
      "259205",
      "211665",
      "27656",
      "203327",
      "228779",
      "73206",
      "20261",
      "184934",
      "80275",
      "247532",
      "198183",
      "112361",
      "78163",
      "72589",
      "214592",
      "273580",
      "73552",
      "200293",
      "103125",
      "7683",
      "34136",
      "275170",
      "285367",
      "177389",
      "48994",
      "103184",
      "260889",
      "260683",
      "138089",
      "51323",
      "19285",
      "152155",
      "240003",
      "246534",
      "96750",
      "89199",
      "204992",
      "245992",
      "238906",
      "205344",
      "42627",
      "174451",
      "162150",
      "87450",
      "20594",
      "890",
      "90026",
      "92970",
      "54363",
      "3551",
      "29516",


In [6]:
blender_script = WORK_DIR / 'phase4_blend.py'
blender_script.write_text('from __future__ import annotations\n\nimport hashlib\nimport json\nimport pickle\nimport sys\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport yaml\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nfrom legalir.fusion import rrf\nfrom legalir.storage import read_json, read_jsonl, write_json\nfrom legalir.text import normalize_question\nfrom legalir.validation import grouped_folds, score_candidates, score_predictions, validate_submission_shape\n\n\nRERANKERS = ("jina", "vietnamese_reranker", "legal_reranker")\nRETRIEVAL_CHANNELS = (\n    "bm25",\n    "accent_char",\n    "vietlegal_harrier",\n    "vietnamese_embedding",\n    "nemotron",\n    "query_memory",\n    "query_exact",\n)\n\n\ndef fuse_cached(\n    retrievals: dict[str, dict[str, Any]],\n    weights: dict[str, float],\n    rrf_k: int,\n    limit: int,\n) -> dict[str, dict[str, list[str]]]:\n    return {\n        qid: {"candidates": rrf(retrieval["channels"], weights, rrf_k, limit)}\n        for qid, retrieval in retrievals.items()\n    }\n\n\ndef engine_ranks(artifacts: Path, split: str, engine: str, fold: int | None = None) -> dict[str, list[str]]:\n    suffix = f"_{fold}" if fold is not None else ""\n    separate = artifacts / f"rerank_{split}{suffix}_{engine}.json"\n    if separate.is_file():\n        return read_json(separate)[engine]\n    combined = artifacts / f"rerank_{split}{suffix}.json"\n    payload = read_json(combined)\n    if engine not in payload:\n        raise RuntimeError(f"{engine} is missing from {combined}")\n    return payload[engine]\n\n\ndef inner_fold(question: str) -> int:\n    # A salt different from grouped_folds is essential: every selected question\n    # is already in outer fold zero under the main fold hash.\n    key = "phase4-inner-v1:" + normalize_question(question)\n    return int(hashlib.sha256(key.encode("utf-8")).hexdigest()[:12], 16) % 5\n\n\ndef reciprocal_rank(rank: int, k: int = 20) -> float:\n    return 1.0 / (k + rank)\n\n\ndef rank_features(rank: int, missing_rank: int) -> list[float]:\n    clipped = min(rank, missing_rank)\n    denominator = max(1, missing_rank - 1)\n    return [\n        reciprocal_rank(clipped),\n        1.0 / clipped,\n        1.0 - min(clipped - 1, denominator) / denominator,\n        float(clipped <= 5),\n        float(clipped <= 10),\n        float(clipped <= 20),\n        float(clipped <= 50),\n        float(clipped < missing_rank),\n    ]\n\n\ndef feature_vector(document: str, orders: dict[str, dict[str, int]]) -> list[float]:\n    features: list[float] = []\n    core_ranks: list[int] = []\n    for name in ("first_stage", *RERANKERS):\n        rank = orders[name].get(document, 81)\n        core_ranks.append(rank)\n        features.extend(rank_features(rank, 81))\n    retrieval_ranks: list[int] = []\n    for name in RETRIEVAL_CHANNELS:\n        rank = orders[name].get(document, 151)\n        retrieval_ranks.append(rank)\n        features.extend(rank_features(rank, 151))\n\n    all_ranks = core_ranks + retrieval_ranks\n    features.extend(\n        [\n            float(sum(rank <= 5 for rank in core_ranks)),\n            float(sum(rank <= 10 for rank in core_ranks)),\n            float(sum(rank <= 20 for rank in core_ranks)),\n            float(sum(rank <= 20 for rank in retrieval_ranks)),\n            float(sum(rank < 151 for rank in retrieval_ranks)),\n            float(min(all_ranks)),\n            float(max(core_ranks)),\n            float(np.mean(core_ranks)),\n            float(np.std(core_ranks)),\n        ]\n    )\n    phase2_score = (\n        0.3 * reciprocal_rank(core_ranks[0])\n        + 0.5 * reciprocal_rank(core_ranks[1])\n        + 0.5 * reciprocal_rank(core_ranks[2])\n    )\n    legal_residual = reciprocal_rank(core_ranks[3]) - reciprocal_rank(core_ranks[0])\n    features.extend([phase2_score, legal_residual])\n    return features\n\n\ndef make_rows(\n    questions: list[dict[str, Any]],\n    fused: dict[str, dict[str, Any]],\n    retrievals: dict[str, dict[str, Any]],\n    rerankings: dict[str, dict[str, list[str]]],\n    labelled: bool,\n) -> tuple[np.ndarray, np.ndarray, list[tuple[str, str]], np.ndarray]:\n    rows: list[list[float]] = []\n    labels: list[int] = []\n    metadata: list[tuple[str, str]] = []\n    groups: list[int] = []\n    for question in questions:\n        qid = question["qid"]\n        candidates = fused[qid]["candidates"][:80]\n        orders = {"first_stage": {doc: rank for rank, doc in enumerate(candidates, 1)}}\n        for name, values in rerankings.items():\n            orders[name] = {doc: rank for rank, doc in enumerate(values[qid], 1)}\n        for name in RETRIEVAL_CHANNELS:\n            orders[name] = {doc: rank for rank, doc in enumerate(retrievals[qid]["channels"][name], 1)}\n        truth = set(question.get("answers", []))\n        group = inner_fold(question["question"])\n        for document in candidates:\n            rows.append(feature_vector(document, orders))\n            labels.append(int(document in truth) if labelled else 0)\n            metadata.append((qid, document))\n            groups.append(group)\n    return (\n        np.asarray(rows, dtype=np.float32),\n        np.asarray(labels, dtype=np.int8),\n        metadata,\n        np.asarray(groups, dtype=np.int8),\n    )\n\n\ndef standardize_per_query(values: np.ndarray, metadata: list[tuple[str, str]]) -> np.ndarray:\n    result = np.zeros(len(values), dtype=np.float64)\n    by_qid: defaultdict[str, list[int]] = defaultdict(list)\n    for index, (qid, _) in enumerate(metadata):\n        by_qid[qid].append(index)\n    for indices in by_qid.values():\n        scores = values[indices]\n        scale = scores.std()\n        result[indices] = (scores - scores.mean()) / (scale if scale > 1e-9 else 1.0)\n    return result\n\n\ndef scores_by_question(values: np.ndarray, metadata: list[tuple[str, str]]) -> dict[str, dict[str, float]]:\n    output: defaultdict[str, dict[str, float]] = defaultdict(dict)\n    for score, (qid, document) in zip(values, metadata, strict=True):\n        output[qid][document] = float(score)\n    return dict(output)\n\n\ndef protected_predictions(\n    blended_scores: np.ndarray,\n    metadata: list[tuple[str, str]],\n    baseline: dict[str, list[str]],\n    margin: float,\n) -> tuple[dict[str, list[str]], dict[str, int]]:\n    per_query = scores_by_question(blended_scores, metadata)\n    output: dict[str, list[str]] = {}\n    promoted_questions = 0\n    promotions = 0\n    for qid, scores in per_query.items():\n        selected = list(baseline[qid])\n        outsiders = [doc for doc, _ in sorted(scores.items(), key=lambda item: (-item[1], item[0])) if doc not in selected]\n        changed = False\n        for outsider in outsiders:\n            weakest = min(selected, key=lambda doc: (scores[doc], doc))\n            if scores[outsider] < scores[weakest] + margin:\n                break\n            selected[selected.index(weakest)] = outsider\n            promotions += 1\n            changed = True\n        if changed:\n            promoted_questions += 1\n        # Ordering is irrelevant to Recall, but sorting makes the output deterministic.\n        output[qid] = sorted(selected, key=lambda doc: (-scores[doc], doc))\n    return output, {"promoted_questions": promoted_questions, "promotions": promotions}\n\n\ndef new_model(c_value: float):\n    return make_pipeline(\n        StandardScaler(),\n        LogisticRegression(\n            C=c_value,\n            class_weight="balanced",\n            max_iter=600,\n            solver="liblinear",\n            random_state=2026,\n        ),\n    )\n\n\ndef baseline_predictions(\n    questions: list[dict[str, Any]],\n    fused: dict[str, dict[str, Any]],\n    rerankings: dict[str, dict[str, list[str]]],\n    final_weights: dict[str, Any],\n) -> dict[str, list[str]]:\n    return {\n        question["qid"]: rrf(\n            {\n                "first_stage": fused[question["qid"]]["candidates"],\n                "jina": rerankings["jina"][question["qid"]],\n                "vietnamese_reranker": rerankings["vietnamese_reranker"][question["qid"]],\n            },\n            final_weights["weights"],\n            final_weights["rrf_k"],\n            5,\n        )\n        for question in questions\n    }\n\n\ndef main() -> None:\n    work = Path(sys.argv[1])\n    config = yaml.safe_load(Path(sys.argv[2]).read_text(encoding="utf-8"))\n    artifacts = Path(config["paths"]["artifacts_dir"])\n    if not artifacts.is_absolute():\n        artifacts = work / artifacts\n\n    first_stage = read_json(artifacts / "first_stage_weights.json")\n    phase2_final = read_json(artifacts / "phase2_final_weights.json")\n    retrieval_train = read_json(artifacts / "retrieval_train.json")\n    retrieval_public = read_json(artifacts / "retrieval_public.json")\n    fused_limit = int(config["retrieval"]["fused_top_k"])\n    fused_train = fuse_cached(retrieval_train, first_stage["weights"], first_stage["rrf_k"], fused_limit)\n    fused_public = fuse_cached(retrieval_public, first_stage["weights"], first_stage["rrf_k"], fused_limit)\n    train_questions = list(read_jsonl(artifacts / "train_questions.jsonl"))\n    public_questions = list(read_jsonl(artifacts / "public_questions.jsonl"))\n    outer_folds = grouped_folds(train_questions, config["validation"]["folds"])\n    fold_questions = [question for question in train_questions if outer_folds[question["qid"]] == 0]\n\n    train_rerankings = {name: engine_ranks(artifacts, "train", name, 0) for name in RERANKERS}\n    public_rerankings = {name: engine_ranks(artifacts, "public", name) for name in RERANKERS}\n    for questions, rerankings, split in (\n        (fold_questions, train_rerankings, "train"),\n        (public_questions, public_rerankings, "public"),\n    ):\n        expected = {question["qid"] for question in questions}\n        for name, values in rerankings.items():\n            missing = expected.difference(values)\n            wrong_depth = [qid for qid in expected.intersection(values) if len(values[qid]) != 80]\n            if missing or wrong_depth:\n                raise RuntimeError(f"{split}/{name}: missing={len(missing)}, non_top80={len(wrong_depth)}")\n\n    baseline = baseline_predictions(fold_questions, fused_train, train_rerankings, phase2_final)\n    baseline_metrics = score_predictions(baseline, fold_questions)\n    if any(not set(baseline[q["qid"]]).issubset(fused_train[q["qid"]]["candidates"][:80]) for q in fold_questions):\n        raise RuntimeError("Phase 2 baseline selected a document outside top 80")\n\n    x_train, y_train, train_metadata, inner_groups = make_rows(\n        fold_questions, fused_train, retrieval_train, train_rerankings, True\n    )\n    if y_train.sum() == 0 or set(inner_groups) != set(range(5)):\n        raise RuntimeError(\n            f"Invalid training sample: positives={int(y_train.sum())}, inner_folds={sorted(set(inner_groups))}"\n        )\n    anchor_train = standardize_per_query(x_train[:, -2].astype(np.float64), train_metadata)\n    baseline_per_inner = {\n        str(fold): score_predictions(\n            baseline,\n            [question for question in fold_questions if inner_fold(question["question"]) == fold],\n        )\n        for fold in range(5)\n    }\n\n    trials: list[dict[str, Any]] = []\n    for c_value in (0.03, 0.1, 0.3, 1.0, 3.0):\n        oof = np.zeros(len(y_train), dtype=np.float64)\n        for heldout in range(5):\n            train_mask = inner_groups != heldout\n            valid_mask = inner_groups == heldout\n            model = new_model(c_value)\n            model.fit(x_train[train_mask], y_train[train_mask])\n            oof[valid_mask] = model.decision_function(x_train[valid_mask])\n        learned = standardize_per_query(oof, train_metadata)\n        for alpha in (0.35, 0.5, 0.65, 0.8, 1.0):\n            blended = alpha * learned + (1.0 - alpha) * anchor_train\n            for margin in (0.0, 0.25, 0.5, 0.75, 1.0):\n                prediction, promotion = protected_predictions(blended, train_metadata, baseline, margin)\n                per_inner = {\n                    str(fold): score_predictions(\n                        prediction,\n                        [question for question in fold_questions if inner_fold(question["question"]) == fold],\n                    )\n                    for fold in range(5)\n                }\n                deltas = [per_inner[str(fold)]["recall"] - baseline_per_inner[str(fold)]["recall"] for fold in range(5)]\n                trials.append(\n                    {\n                        "C": c_value,\n                        "alpha": alpha,\n                        "promotion_margin": margin,\n                        "metrics": score_predictions(prediction, fold_questions),\n                        "per_inner": per_inner,\n                        "nonnegative_inner_folds": sum(delta >= -1e-12 for delta in deltas),\n                        "worst_inner_recall_delta": min(deltas),\n                        **promotion,\n                    }\n                )\n\n    robust = [\n        row\n        for row in trials\n        if row["nonnegative_inner_folds"] >= 4 and row["worst_inner_recall_delta"] >= -0.005\n    ]\n    pool = robust or trials\n    best = max(\n        pool,\n        key=lambda row: (\n            row["metrics"]["recall"],\n            row["nonnegative_inner_folds"],\n            row["worst_inner_recall_delta"],\n            row["metrics"]["precision"],\n            -row["promotions"],\n            -row["C"],\n        ),\n    )\n\n    final_model = new_model(best["C"])\n    final_model.fit(x_train, y_train)\n    x_public, _, public_metadata, _ = make_rows(\n        public_questions, fused_public, retrieval_public, public_rerankings, False\n    )\n    learned_public = standardize_per_query(final_model.decision_function(x_public), public_metadata)\n    anchor_public = standardize_per_query(x_public[:, -2].astype(np.float64), public_metadata)\n    baseline_public = baseline_predictions(public_questions, fused_public, public_rerankings, phase2_final)\n    blended_public = best["alpha"] * learned_public + (1.0 - best["alpha"]) * anchor_public\n    final_prediction, private_promotion = protected_predictions(\n        blended_public, public_metadata, baseline_public, best["promotion_margin"]\n    )\n\n    corpus_ids = {row["doc_id"] for row in read_jsonl(artifacts / "corpus.jsonl")}\n    submission = {qid: {"answer": documents} for qid, documents in final_prediction.items()}\n    validate_submission_shape(submission, public_questions, corpus_ids)\n    submission_path = work / "submission_phase4_private_final.json"\n    write_json(submission_path, submission)\n    with (work / "phase4_blender.pkl").open("wb") as handle:\n        pickle.dump(final_model, handle)\n\n    report = {\n        "experiment_id": "phase4-phase2-legal-bge-protected-blender",\n        "training_questions": len(fold_questions),\n        "training_pairs": len(y_train),\n        "positive_pairs": int(y_train.sum()),\n        "feature_count": int(x_train.shape[1]),\n        "inner_fold_counts": {\n            str(fold): int(sum(inner_fold(question["question"]) == fold for question in fold_questions))\n            for fold in range(5)\n        },\n        "candidate_top_k": 80,\n        "candidate_recall_fold0": score_candidates(\n            {question["qid"]: fused_train[question["qid"]]["candidates"][:80] for question in fold_questions},\n            fold_questions,\n        )["candidate_recall"],\n        "phase2_baseline_fold0": baseline_metrics,\n        "phase2_baseline_per_inner": baseline_per_inner,\n        "selected_cross_fitted": best,\n        "recall_delta": best["metrics"]["recall"] - baseline_metrics["recall"],\n        "robust_trials": len(robust),\n        "tested_configurations": len(trials),\n        "private_promotions": private_promotion,\n        "submission": str(submission_path),\n    }\n    write_json(work / "phase4_report.json", report)\n    print(json.dumps(report, ensure_ascii=False, indent=2))\n\n\nif __name__ == "__main__":\n    main()', encoding='utf-8')
run(sys.executable, blender_script, WORK_DIR, config_path, cwd=WORK_DIR, env=runtime_env)
submission_json = WORK_DIR / 'submission_phase4_private_final.json'
submission_zip = WORK_DIR / 'submission_phase4_private_final.zip'
run('zip', '-j', submission_zip, submission_json)
print('PHASE 4 FINAL:', submission_zip)


+ /usr/bin/python3 /kaggle/working/legalir-phase4-private-final/phase4_blend.py /kaggle/working/legalir-phase4-private-final /kaggle/working/legalir-phase4-private-final/kaggle_rtx_pro_6000_phase4.yaml
{
  "experiment_id": "phase4-phase2-legal-bge-protected-blender",
  "training_questions": 1345,
  "training_pairs": 107600,
  "positive_pairs": 1430,
  "feature_count": 99,
  "inner_fold_counts": {
    "0": 254,
    "1": 274,
    "2": 252,
    "3": 273,
    "4": 292
  },
  "candidate_top_k": 80,
  "candidate_recall_fold0": 0.9876703841387857,
  "phase2_baseline_fold0": {
    "recall": 0.9351301115241636,
    "precision": 0.1986617100371747
  },
  "phase2_baseline_per_inner": {
    "0": {
      "recall": 0.9429133858267716,
      "precision": 0.20236220472440947
    },
    "1": {
      "recall": 0.944647201946472,
      "precision": 0.2
    },
    "2": {
      "recall": 0.9378306878306879,
      "precision": 0.19603174603174606
    },
    "3": {
      "recall": 0.9255189255189255,
      "

In [7]:
report = json.loads((WORK_DIR / 'phase4_report.json').read_text(encoding='utf-8'))
model_manifest = json.loads((artifacts / 'model_manifest.json').read_text(encoding='utf-8'))
submission = json.loads((WORK_DIR / 'submission_phase4_private_final.json').read_text(encoding='utf-8'))
phase2_submission_path = PHASE2_ARTIFACTS.parent / 'submission_phase2_private_harrier.json'
comparison = None
if phase2_submission_path.is_file():
    phase2_submission = json.loads(phase2_submission_path.read_text(encoding='utf-8'))
    shared = set(submission).intersection(phase2_submission)
    comparison = {
        'questions_compared': len(shared),
        'changed_answer_sets': sum(set(submission[q]['answer']) != set(phase2_submission[q]['answer']) for q in shared),
        'changed_top1': sum(submission[q]['answer'][0] != phase2_submission[q]['answer'][0] for q in shared),
    }
report.update({
    'test_file': TEST_FILENAME,
    'test_questions': test_question_count,
    'test_sha256': test_sha256,
    'project_commit': delta_manifest['project_commit'],
    'models': model_manifest['models'],
    'total_parameters': model_manifest['total_parameters'],
    'private_comparison_to_phase2': comparison,
    'submission_zip': str(WORK_DIR / 'submission_phase4_private_final.zip'),
})
(WORK_DIR / 'phase4_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(report, ensure_ascii=False, indent=2))


{
  "experiment_id": "phase4-phase2-legal-bge-protected-blender",
  "training_questions": 1345,
  "training_pairs": 107600,
  "positive_pairs": 1430,
  "feature_count": 99,
  "inner_fold_counts": {
    "0": 254,
    "1": 274,
    "2": 252,
    "3": 273,
    "4": 292
  },
  "candidate_top_k": 80,
  "candidate_recall_fold0": 0.9876703841387857,
  "phase2_baseline_fold0": {
    "recall": 0.9351301115241636,
    "precision": 0.1986617100371747
  },
  "phase2_baseline_per_inner": {
    "0": {
      "recall": 0.9429133858267716,
      "precision": 0.20236220472440947
    },
    "1": {
      "recall": 0.944647201946472,
      "precision": 0.2
    },
    "2": {
      "recall": 0.9378306878306879,
      "precision": 0.19603174603174606
    },
    "3": {
      "recall": 0.9255189255189255,
      "precision": 0.19633699633699633
    },
    "4": {
      "recall": 0.9260844748858448,
      "precision": 0.19863013698630136
    }
  },
  "selected_cross_fitted": {
    "C": 0.1,
    "alpha": 1.0,
    "